In [ ]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao


,Ano,Base,Tabelas
0,2025,2025.duckdb,8
1,2026,2026.duckdb,11


In [ ]:
# ==========================
# 8. Ler dados das bases
# ==========================

df = pd.concat(
    [
        CON[ano].execute(
            f"""
            SELECT *
            FROM inform_27_{ano}
            WHERE Gestion = ?
            """,
            [GESTAO]
        ).df().assign(ANO=ano)
        for ano in ANOS
    ],
    ignore_index=True
)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

display(df.head())

Linhas: 238,314
Colunas: 59


,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem,ANO
0,46355,PTG,DL Transportes Fernando Simões Monteiro Unip. ...,Transportes Fernando Simões Monteiro Unip. Lda,0000XXX,0000XXX,11.82,11.25,0.57,0.31,...,AUCHAN VISEU,TAM,EUR,Trailer 33 plts,33,DIESEL,135.191,NO,SAL_DAT027_2025_01_07.xls,2025
1,46356,PTG,SALVESEN LOGISTICA PORTUGAL / CARMONTI-IND.CAR...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,11.82,2.8,9.02,0.31,...,DL Transportes Fernando Simões Monteiro Unip. Lda,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027_2025_01_07.xls,2025
2,46357,PTG,SALVESEN LOGISTICA PORTUGAL / CARMONTI-IND.CAR...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0,0.27,-0.27,0.03,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027_2025_01_07.xls,2025
3,46358,PTG,SALVESEN LOGISTICA PORTUGAL / CARMONTI-IND.CAR...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,11.02,18.07,-7.06,2,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027_2025_01_07.xls,2025
4,46359,CEP,SALVESEN LOGISTICA PORTUGAL / CARMONTI-IND.CAR...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,24.36,9.04,15.32,1,...,DL Transportes Fernando Simões Monteiro Unip. Lda,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027_2025_01_07.xls,2025


In [ ]:
# ==========================
# 9. Paletes por cliente e distrito
# ==========================

import unicodedata
import pandas as pd

# ==========================
# 1. Parâmetros
# ==========================

MES = 8  # Agosto (1 a 12)

# Utiliza ANOS, GESTAO e CON já definidos anteriormente


# ==========================
# 2. Distritos
# ==========================

distritos = [
    "Aveiro",
    "Beja",
    "Braga",
    "Bragança",
    "Castelo Branco",
    "Coimbra",
    "Évora",
    "Faro",
    "Guarda",
    "Leiria",
    "Lisboa",
    "Portalegre",
    "Porto",
    "Santarém",
    "Setúbal",
    "Viana do Castelo",
    "Vila Real",
    "Viseu"
]


# ==========================
# 3. Mapeamento
# ==========================

grupos = {
    "Aveiro": [
        "Aveiro", "Albergaria-a-Velha", "Anadia",
        "Arouca", "Espinho", "Estarreja",
        "Mealhada", "Murtosa", "Oliveira de Azeméis",
        "Oliveira do Bairro", "Ovar",
        "Santa Maria da Feira", "Sever do Vouga",
        "São João da Madeira", "Vale de Cambra",
        "Águeda", "Ílhavo"
    ],
    "Beja": ["Beja", "Castro Verde"],
    "Braga": ["Braga"],
    "Bragança": ["Bragança"],
    "Castelo Branco": ["Castelo Branco"],
    "Coimbra": ["Coimbra"],
    "Évora": ["Évora"],
    "Faro": ["Faro"],
    "Guarda": ["Guarda"],
    "Leiria": ["Leiria"],
    "Lisboa": ["Lisboa"],
    "Portalegre": ["Portalegre"],
    "Porto": ["Porto"],
    "Santarém": ["Santarém", "Ourém"],
    "Setúbal": ["Setúbal"],
    "Viana do Castelo": ["Viana do Castelo"],
    "Vila Real": ["Vila Real"],
    "Viseu": ["Viseu"]
}


# ==========================
# 4. Normalização de texto
# ==========================

def normalizar(texto):
    if pd.isna(texto):
        return ""

    texto = unicodedata.normalize(
        "NFKD", str(texto).strip()
    )

    return "".join(
        c for c in texto
        if not unicodedata.combining(c)
    ).upper()


mapa = {
    normalizar(localidade): distrito
    for distrito, localidades in grupos.items()
    for localidade in localidades
}


# ==========================
# 5. Ler dados - Agosto 2026
# ==========================

ANO = 2026
MES = 8

dados = CON[ANO].execute(
    f"""
    SELECT
        CODACT,
        ACTIVIDAD,
        PROV_DESTINO,
        PALETS
    FROM inform_27_{ANO}
    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
    """,
    [GESTAO, f"{ANO}{MES:02d}"]
).df()

dados["ANO"] = ANO


# ==========================
# 6. Preparar dados
# ==========================

dados["CODACT"] = (
    dados["CODACT"]
    .astype("string")
    .str.strip()
    .str.zfill(3)
)

dados["PALETS"] = pd.to_numeric(
    dados["PALETS"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

dados["DISTRITO"] = (
    dados["PROV_DESTINO"]
    .map(normalizar)
    .map(mapa)
)


# ==========================
# 7. Validar não mapeados
# ==========================

nao_mapeadas = (
    dados[dados["DISTRITO"].isna()]
    .groupby(
        ["ANO", "PROV_DESTINO"],
        dropna=False
    )["PALETS"]
    .sum()
    .reset_index()
)


# ==========================
# 8. Tabela dinâmica
# ==========================

df_volumes = pd.pivot_table(
    dados[dados["DISTRITO"].notna()],
    index=["ANO", "CODACT", "ACTIVIDAD"],
    columns="DISTRITO",
    values="PALETS",
    aggfunc="sum",
    fill_value=0
)

# Garantir as 18 colunas
df_volumes = df_volumes.reindex(
    columns=distritos,
    fill_value=0
)

# Total por cliente
df_volumes["TOTAL"] = df_volumes[distritos].sum(axis=1)

df_volumes = df_volumes.reset_index()
df_volumes.columns.name = None


# ==========================
# 9. Resultados
# ==========================

print(f"Mês analisado: {MES:02d}")
print(f"Anos: {ANOS}")
print(f"Registos analisados: {len(dados):,}")

print("\nPALETES POR CLIENTE E DISTRITO")

display(df_volumes)

print("\nPROVÍNCIAS NÃO MAPEADAS")

display(nao_mapeadas)

Mês analisado: 08
Anos: [2025, 2026]
Registos analisados: 16,301

PALETES POR CLIENTE E DISTRITO


,ANO,CODACT,ACTIVIDAD,Aveiro,Beja,Braga,Bragança,Castelo Branco,Coimbra,Évora,...,Leiria,Lisboa,Portalegre,Porto,Santarém,Setúbal,Viana do Castelo,Vila Real,Viseu,TOTAL
0,2026,011,DANONE PORTUGAL,405.0,41.0,122.0,81.0,84.0,102.0,5.0,...,50.0,8302.0,17.0,7079.0,668.0,377.0,70.0,1.0,0.0,18324.0
1,2026,013,DANONE PORTUGAL CAPILAR,271.0,2.0,259.0,0.0,0.0,41.0,0.0,...,42.0,118.0,0.0,270.0,34.0,62.0,4.0,0.0,12.0,1120.0
2,2026,015,TROPICANA,8.0,0.0,0.0,0.0,0.0,10.0,0.0,...,0.0,123.0,0.0,51.0,6.0,0.0,0.0,0.0,0.0,216.0
3,2026,028,PLN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,21.0,0.0,0.0,0.0,0.0,0.0,25.0
4,2026,074,NUTRICIA,160.0,41.0,196.0,26.0,36.0,251.0,22.0,...,43.0,1336.0,5.0,1418.0,154.0,130.0,41.0,27.0,38.0,4128.0
5,2026,084,DANONE PORTUGAL TLD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,528.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,528.0
6,2026,116,ORANGINA SCHWEPPES ESPAÑA BASE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
7,2026,117,ORANGINA SCHWEPPES,28.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,26.0,0.0,11.0,0.0,21.0,0.0,0.0,0.0,159.0
8,2026,118,ORANGNA SCHWEPPES PORTUGAL,48.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,234.0,0.0,176.0,66.0,35.0,0.0,0.0,0.0,577.0
9,2026,131,RANA,0.0,0.0,0.0,0.0,0.0,14.0,0.0,...,0.0,663.0,0.0,338.0,88.0,32.0,0.0,0.0,0.0,1219.0



PROVÍNCIAS NÃO MAPEADAS


,ANO,PROV_DESTINO,PALETS
0,2026,,631.0
1,2026,AÇORES,27.0
2,2026,Madrid,732.0
3,2026,NaN,1512.5


In [ ]:
from pathlib import Path

ficheiro = Path.home() / "Documents" / "Paletes_Distritos_Agosto_2026.xlsx"

df_volumes.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

Excel exportado: C:\Users\LISARR\Documents\Paletes_Distritos_Agosto_2026.xlsx


In [ ]:
# ==========================
# 10. Ingressos por cliente e distrito
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

# 1. Ler dados
dados_ingresso = CON[ANO].execute(
    f"""
    SELECT
        CODACT,
        ACTIVIDAD,
        PROV_DESTINO,
        INGRESODT
    FROM inform_27_{ANO}
    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
    """,
    [GESTAO, f"{ANO}{MES:02d}"]
).df()

# 2. Preparar dados
dados_ingresso["CODACT"] = (
    dados_ingresso["CODACT"]
    .astype("string")
    .str.strip()
    .str.zfill(3)
)

dados_ingresso["INGRESODT"] = pd.to_numeric(
    dados_ingresso["INGRESODT"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# 3. Normalizar distritos
# Reutiliza 'normalizar', 'mapa' e 'distritos'
# definidos na célula anterior

dados_ingresso["DISTRITO"] = (
    dados_ingresso["PROV_DESTINO"]
    .map(normalizar)
    .map(mapa)
)

# 4. Tabela dinâmica
df_ingressos = pd.pivot_table(
    dados_ingresso[dados_ingresso["DISTRITO"].notna()],
    index=["CODACT", "ACTIVIDAD"],
    columns="DISTRITO",
    values="INGRESODT",
    aggfunc="sum",
    fill_value=0
)

# Garantir os 18 distritos
df_ingressos = df_ingressos.reindex(
    columns=distritos,
    fill_value=0
)

# Total por cliente
df_ingressos["TOTAL"] = df_ingressos[distritos].sum(axis=1)

df_ingressos = df_ingressos.reset_index()
df_ingressos.columns.name = None

# Arredondar valores
df_ingressos[distritos + ["TOTAL"]] = (
    df_ingressos[distritos + ["TOTAL"]].round(2)
)

# 5. Validar províncias não mapeadas
nao_mapeadas_ingresso = (
    dados_ingresso[dados_ingresso["DISTRITO"].isna()]
    .groupby("PROV_DESTINO", dropna=False)["INGRESODT"]
    .sum()
    .reset_index()
)

# 6. Visualizar
display(df_ingressos)

print("Províncias não mapeadas:")
display(nao_mapeadas_ingresso)

# 7. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Ingressos_Distritos_Agosto_2026.xlsx"
)

df_ingressos.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

,CODACT,ACTIVIDAD,Aveiro,Beja,Braga,Bragança,Castelo Branco,Coimbra,Évora,Faro,...,Leiria,Lisboa,Portalegre,Porto,Santarém,Setúbal,Viana do Castelo,Vila Real,Viseu,TOTAL
0,011,DANONE PORTUGAL,1703.28,2435.78,514.16,938.99,1240.66,0.0,0.0,11747.8,...,737.79,18758.36,1048.75,26677.37,9212.75,5694.69,423.92,0.0,0.0,81134.3
1,013,DANONE PORTUGAL CAPILAR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,015,TROPICANA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,028,PLN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,171.14,0.0,0.0,0.0,0.0,0.0,171.14
4,074,NUTRICIA,1248.98,697.86,1732.7,703.6,678.65,2785.07,548.36,3621.37,...,770.73,15164.11,112.2,16384.54,3633.38,1850.09,397.8,452.49,548.82,51825.87
5,084,DANONE PORTUGAL TLD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2271.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2271.52
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,117,ORANGINA SCHWEPPES,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,118,ORANGNA SCHWEPPES PORTUGAL,3236.96,0.0,0.0,0.0,0.0,0.0,0.0,1088.47,...,0.0,8752.17,0.0,10168.5,2936.81,1239.59,0.0,0.0,0.0,27422.5
9,131,RANA,0.0,0.0,0.0,0.0,0.0,169.12,0.0,2283.07,...,0.0,7383.15,0.0,7451.52,1516.76,1035.19,0.0,0.0,0.0,19838.81


Províncias não mapeadas:


,PROV_DESTINO,INGRESODT
0,,348.59
1,AÇORES,2095.88
2,Madrid,13200.0
3,NaN,547.23


Excel exportado: C:\Users\LISARR\Documents\Ingressos_Distritos_Agosto_2026.xlsx


In [ ]:
# ==========================
# 11. Custos por cliente e distrito
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

# 1. Ler dados
dados_custo = CON[ANO].execute(
    f"""
    SELECT
        CODACT,
        ACTIVIDAD,
        PROV_DESTINO,
        COSTEDT
    FROM inform_27_{ANO}
    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
    """,
    [GESTAO, f"{ANO}{MES:02d}"]
).df()

# 2. Preparar dados
dados_custo["CODACT"] = (
    dados_custo["CODACT"]
    .astype("string")
    .str.strip()
    .str.zfill(3)
)

dados_custo["COSTEDT"] = pd.to_numeric(
    dados_custo["COSTEDT"]
    .astype("string")
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

# 3. Normalizar distritos
# Reutiliza 'normalizar', 'mapa' e 'distritos'
# definidos anteriormente

dados_custo["DISTRITO"] = (
    dados_custo["PROV_DESTINO"]
    .map(normalizar)
    .map(mapa)
)

# 4. Tabela dinâmica
df_custos = pd.pivot_table(
    dados_custo[dados_custo["DISTRITO"].notna()],
    index=["CODACT", "ACTIVIDAD"],
    columns="DISTRITO",
    values="COSTEDT",
    aggfunc="sum",
    fill_value=0
)

# Garantir os 18 distritos
df_custos = df_custos.reindex(
    columns=distritos,
    fill_value=0
)

# Total por cliente
df_custos["TOTAL"] = df_custos[distritos].sum(axis=1)

df_custos = df_custos.reset_index()
df_custos.columns.name = None

# Arredondar valores
df_custos[distritos + ["TOTAL"]] = (
    df_custos[distritos + ["TOTAL"]].round(2)
)

# 5. Validar províncias não mapeadas
nao_mapeadas_custo = (
    dados_custo[dados_custo["DISTRITO"].isna()]
    .groupby("PROV_DESTINO", dropna=False)["COSTEDT"]
    .sum()
    .reset_index()
)

# 6. Visualizar
display(df_custos)

print("Províncias não mapeadas:")
display(nao_mapeadas_custo)

# 7. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Custos_Distritos_Agosto_2026.xlsx"
)

df_custos.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

,CODACT,ACTIVIDAD,Aveiro,Beja,Braga,Bragança,Castelo Branco,Coimbra,Évora,Faro,...,Leiria,Lisboa,Portalegre,Porto,Santarém,Setúbal,Viana do Castelo,Vila Real,Viseu,TOTAL
0,011,DANONE PORTUGAL,7994.09,2251.39,5042.91,2982.41,2863.85,450.64,6.72,16946.35,...,1618.37,29835.64,930.37,71518.06,8917.65,5315.53,2831.57,20.27,0.0,159525.82
1,013,DANONE PORTUGAL CAPILAR,5335.44,22.69,6228.1,0.0,0.0,398.28,0.0,3.34,...,487.12,2331.77,0.0,5098.01,452.2,1583.98,177.1,0.0,109.54,22359.52
2,015,TROPICANA,398.7,0.0,0.0,0.0,0.0,154.83,0.0,434.53,...,0.0,1577.93,0.0,849.83,60.69,0.0,0.0,0.0,0.0,3476.51
3,028,PLN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,255.52,0.0,353.81,0.0,0.0,0.0,0.0,0.0,609.33
4,074,NUTRICIA,2126.99,1027.4,3058.63,574.6,457.96,2725.62,320.32,1984.67,...,596.84,13646.08,85.22,19082.72,1949.7,2512.49,463.39,527.81,550.07,51977.48
5,084,DANONE PORTUGAL TLD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2700.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2700.0
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,54.51,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54.51
7,117,ORANGINA SCHWEPPES,302.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1253.27,0.0,179.26,0.0,236.31,0.0,0.0,0.0,3540.34
8,118,ORANGNA SCHWEPPES PORTUGAL,1003.9,0.0,0.0,0.0,0.0,0.0,0.0,256.05,...,0.0,2140.45,0.0,2663.27,1220.37,379.04,0.0,0.0,0.0,7663.08
9,131,RANA,0.0,0.0,0.0,0.0,0.0,4.15,0.0,1326.25,...,0.0,4220.15,0.0,3370.18,1080.48,817.61,0.0,0.0,0.0,10818.82


Províncias não mapeadas:


,PROV_DESTINO,COSTEDT
0,,10905.73
1,AÇORES,1222.68
2,Madrid,12247.52
3,NaN,24812.38


Excel exportado: C:\Users\LISARR\Documents\Custos_Distritos_Agosto_2026.xlsx


In [ ]:
# ==========================
# 12. Resumo por cliente
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

# 1. Extrair dados
df_resumo = CON[ANO].execute(
    f"""
    SELECT
        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,
        ACTIVIDAD,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(INGRESODT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS INGRESODT,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(COSTEDT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS COSTEDT,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(PALETS AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS PALETS

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?

    GROUP BY CODACT, ACTIVIDAD
    ORDER BY CODACT, ACTIVIDAD
    """,
    [GESTAO, f"{ANO}{MES:02d}"]
).df()

# 2. Visualizar
display(df_resumo)

# 3. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Resumo_Clientes_Agosto_2026.xlsx"
)

df_resumo.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

,CODACT,ACTIVIDAD,INGRESODT,COSTEDT,PALETS
0,011,DANONE PORTUGAL,81139.92,159709.89,18342.0
1,013,DANONE PORTUGAL CAPILAR,0.00,56853.64,3046.0
2,015,TROPICANA,0.00,3623.18,234.0
3,028,PLN,171.14,609.33,25.0
4,074,NUTRICIA,51825.87,52335.98,4181.0
5,084,DANONE PORTUGAL TLD,2271.52,2700.00,528.0
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,0.00,54.51,8.0
7,117,ORANGINA SCHWEPPES,0.00,4413.41,170.0
8,118,ORANGNA SCHWEPPES PORTUGAL,29443.52,7980.24,588.0
9,131,RANA,19854.40,10823.02,1220.0


Excel exportado: C:\Users\LISARR\Documents\Resumo_Clientes_Agosto_2026.xlsx


In [ ]:
# ==========================
# 13. Destinos por cliente
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

CLIENTES = ['391', '404', '405', '407', '415', '551', '680', '757']

# 1. Extrair e agregar dados
df_destinos = CON[ANO].execute(
    f"""
    SELECT
        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,
        ACTIVIDAD,
        DESTINO,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(PALETS AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS PALETS,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(INGRESODT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS INGRESODT,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(COSTEDT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS COSTEDT

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
      AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0')
          IN ({','.join(['?'] * len(CLIENTES))})

    GROUP BY 1, 2, 3

    ORDER BY CODACT, PALETS DESC
    """,
    [GESTAO, f"{ANO}{MES:02d}", *CLIENTES]
).df()

# 2. Visualizar todos os destinos
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(df_destinos)

# 3. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Destinos_Clientes_Agosto_2026.xlsx"
)

df_destinos.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

,CODACT,ACTIVIDAD,DESTINO,PALETS,INGRESODT,COSTEDT
0,391,GL IMPORTAÇÃO E EXPORTAÇÃO,SALVESEN LOGISTICA PORTUGAL / GL Importação E ...,525.0,0.00,537.12
1,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL PALMELA / GL Importação E Exportação (391),118.0,2261.50,2209.91
2,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL TORRES NOVAS / GL Importação E Exportação...,103.0,2034.90,2083.85
3,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL SANTO TIRSO / GL Importação E Exportação ...,102.0,3124.28,4127.91
4,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL LOURES / GL Importação E Exportação (391),97.0,1891.32,1722.02
5,391,GL IMPORTAÇÃO E EXPORTAÇÃO,ITMP ALCANENA / GL Importação E Exportação (391),82.0,1435.26,647.49
6,391,GL IMPORTAÇÃO E EXPORTAÇÃO,ITMP PAÇOS FERREIRA / GL Importação E Exportaç...,36.0,1220.44,583.67
7,391,GL IMPORTAÇÃO E EXPORTAÇÃO,PETRÓLEOS DE PORTUGAL - PETROG / GL Importação...,24.0,629.96,1487.39
8,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL & CIA - SANTO TIRSO / Carmonti (403),9.0,147.27,284.82
9,391,GL IMPORTAÇÃO E EXPORTAÇÃO,NaN,6.0,95.16,51.38


Excel exportado: C:\Users\LISARR\Documents\Destinos_Clientes_Agosto_2026.xlsx


In [ ]:
# ==========================
# 13. Destinos por cliente e distrito
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

CLIENTES = ['391', '404', '405', '407', '415', '551', '680', '757']

# 1. Extrair dados
dados_destinos = CON[ANO].execute(
    f"""
    SELECT
        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,
        ACTIVIDAD,
        DESTINO,
        PROV_DESTINO,

        SUM(
            TRY_CAST(REPLACE(CAST(PALETS AS VARCHAR), ',', '.') AS DOUBLE)
        ) AS PALETS,

        SUM(
            TRY_CAST(REPLACE(CAST(INGRESODT AS VARCHAR), ',', '.') AS DOUBLE)
        ) AS INGRESODT,

        SUM(
            TRY_CAST(REPLACE(CAST(COSTEDT AS VARCHAR), ',', '.') AS DOUBLE)
        ) AS COSTEDT

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
      AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0')
          IN ({','.join(['?'] * len(CLIENTES))})

    GROUP BY 1, 2, 3, 4
    """,
    [GESTAO, f"{ANO}{MES:02d}", *CLIENTES]
).df()

# 2. Normalizar distrito
# Reutiliza as variáveis 'normalizar' e 'mapa'

dados_destinos["DISTRITO"] = (
    dados_destinos["PROV_DESTINO"]
    .map(normalizar)
    .map(mapa)
    .fillna("NÃO MAPEADO")
)

# 3. Agrupar por cliente, destino e distrito
df_destinos = (
    dados_destinos
    .groupby(
        ["CODACT", "ACTIVIDAD", "DESTINO", "DISTRITO"],
        dropna=False,
        as_index=False
    )[["PALETS", "INGRESODT", "COSTEDT"]]
    .sum()
)

# 4. Arredondar valores
df_destinos[["PALETS", "INGRESODT", "COSTEDT"]] = (
    df_destinos[["PALETS", "INGRESODT", "COSTEDT"]].round(2)
)

# 5. Ordenar por cliente e paletes (decrescente)
df_destinos = df_destinos.sort_values(
    ["CODACT", "PALETS"],
    ascending=[True, False]
).reset_index(drop=True)

# 6. Visualizar
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(df_destinos)

# 7. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Destinos_Clientes_Distritos_Agosto_2026.xlsx"
)

df_destinos.to_excel(ficheiro, index=False)

print(f"Excel exportado: {ficheiro}")

,CODACT,ACTIVIDAD,DESTINO,DISTRITO,PALETS,INGRESODT,COSTEDT
0,391,GL IMPORTAÇÃO E EXPORTAÇÃO,SALVESEN LOGISTICA PORTUGAL / GL Importação E ...,Lisboa,525.0,0.00,537.12
1,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL PALMELA / GL Importação E Exportação (391),Setúbal,118.0,2261.50,2209.91
2,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL TORRES NOVAS / GL Importação E Exportação...,Santarém,103.0,2034.90,2083.85
3,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL SANTO TIRSO / GL Importação E Exportação ...,Porto,102.0,3124.28,4127.91
4,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL LOURES / GL Importação E Exportação (391),Lisboa,97.0,1891.32,1722.02
5,391,GL IMPORTAÇÃO E EXPORTAÇÃO,ITMP ALCANENA / GL Importação E Exportação (391),Santarém,82.0,1435.26,647.49
6,391,GL IMPORTAÇÃO E EXPORTAÇÃO,ITMP PAÇOS FERREIRA / GL Importação E Exportaç...,Porto,36.0,1220.44,583.67
7,391,GL IMPORTAÇÃO E EXPORTAÇÃO,PETRÓLEOS DE PORTUGAL - PETROG / GL Importação...,Aveiro,16.0,307.16,972.24
8,391,GL IMPORTAÇÃO E EXPORTAÇÃO,LIDL & CIA - SANTO TIRSO / Carmonti (403),Porto,9.0,147.27,284.82
9,391,GL IMPORTAÇÃO E EXPORTAÇÃO,NaN,Porto,6.0,95.16,51.38


Excel exportado: C:\Users\LISARR\Documents\Destinos_Clientes_Distritos_Agosto_2026.xlsx


In [ ]:
# ==========================
# Entregas Machado e Brites
# Agosto 2026
# ==========================

ANO = 2026
MES = 8

df_entregas = CON[ANO].execute(
    f"""
    SELECT DISTINCT
        FENTREGA,
        CODEUT

    FROM inform_27_{ANO}

    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) = ?
      AND TRANSPORTISTA ILIKE '%MACHADO%'
      AND TRANSPORTISTA ILIKE '%BRITES%'

    ORDER BY FENTREGA, CODEUT
    """,
    [GESTAO, f"{ANO}{MES:02d}"]
).df()

display(df_entregas)

print(f"Total de combinações distintas de data e CODEUT: {len(df_entregas)}")

,FENTREGA,CODEUT
0,20260806,3514019
1,20260807,3514775
2,20260810,3515749
3,20260811,3517262
4,20260811,3518568
5,20260812,3518143
6,20260812,3519571
7,20260813,3519221
8,20260814,3520565
9,20260817,3521477


Total de combinações distintas de data e CODEUT: 15


In [ ]:
# ==========================================
# CUSTOS, INGRESSOS E PALETES POR CLIENTE
# Agosto e Setembro de 2026
# ==========================================

ANO = 2026

CLIENTES = [
    '391', '404', '405', '407', '415',
    '551', '680', '757', '311'
]

df_clientes = CON[ANO].execute(
    f"""
    SELECT
        SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,

        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,

        ACTIVIDAD,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(INGRESODT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS INGRESODT,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(COSTEDT AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS COSTEDT,

        ROUND(SUM(
            TRY_CAST(REPLACE(CAST(PALETS AS VARCHAR), ',', '.') AS DOUBLE)
        ), 2) AS PALETS

    FROM inform_27_{ANO}

    WHERE Gestion = ?

      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
          IN ('202608', '202609')

      AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0')
          IN ({','.join(['?'] * len(CLIENTES))})

    GROUP BY 1, 2, 3

    ORDER BY CODACT, MES
    """,
    [GESTAO, *CLIENTES]
).df()

display(df_clientes)